# In-distribution evaluation of the deep sentiment models (LSTM + DistilBERT)

Gets the **in-distribution** Accuracy / Precision / Recall / F1 for the served sentiment **LSTM** and **DistilBERT** so they can join the eight classical models in Table 7.2 of the thesis.

## Why this notebook exists / methodology
The classical models (Table 7.2) were trained by `train_merged_sentiment_models.py`, which seeds with Python's `random.sample`. The two DL notebooks (`train_lstm_sentiment.ipynb`, `train_distilbert_sentiment.ipynb`) use the **same recipe** (all 50k IMDB + tweets → 500k, 80/20 stratified, `SEED=42`) but sample tweets with **pandas `.sample(random_state=42)`** — a different RNG — so they trained on **different rows**. Running the DL models on the *classical* split would leak ≈¼ of the rows (they'd be in DL training) and inflate the scores.

This notebook therefore reproduces the **DL models' own** held-out 20% split (identical recipe / seed / 90:10 tweet:IMDB distribution as the classical split, just independently sampled rows the DL models never trained on) — **leakage-free**. Both DL notebooks share an identical prep, so one reproduced split evaluates both.

Decision rules match the served app and the classical `.predict` (0.5): LSTM `prob > 0.5`; DistilBERT `argmax`. Metrics match `train_merged_sentiment_models.py` exactly (binary, `pos_label=1`, `zero_division=0`).

## Upload to your Google Drive first (MyDrive root, or edit paths in cell 3)
| File on disk | Size | 
|---|---|
| `data/merged_sentiment.csv` | 182 MB (likely already in Drive from training) |
| `data/sentiment_lstm.onnx` | 25 MB |
| `data/sentiment_lstm_tokenizer.json` | 30 MB |
| `data/distilbert_sentiment_onnx/` (whole folder) | 256 MB |

**Runtime → Change runtime type → GPU** before running.

In [ ]:
# Clean ORT install (avoid CPU/GPU onnxruntime clash) + tokenizer deps
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1
!pip -q install onnxruntime-gpu transformers >/dev/null 2>&1
import onnxruntime as ort
print('onnxruntime', ort.__version__, '| providers:', ort.get_available_providers())
PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
print('Using:', PROVIDERS)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ===== EDIT THESE to match where you uploaded the files in Drive =====
CSV_PATH       = '/content/drive/MyDrive/merged_sentiment.csv'
LSTM_ONNX      = '/content/drive/MyDrive/sentiment_lstm.onnx'
LSTM_TOKENIZER = '/content/drive/MyDrive/sentiment_lstm_tokenizer.json'
DISTILBERT_DIR = '/content/drive/MyDrive/distilbert_sentiment_onnx'
# =====================================================================
import os
for p in [CSV_PATH, LSTM_ONNX, LSTM_TOKENIZER, DISTILBERT_DIR]:
    print(('OK   ' if os.path.exists(p) else 'MISSING  ') + p)

In [ ]:
# Reproduce the DL models' EXACT held-out split (verbatim from the two training notebooks)
import random, numpy as np, pandas as pd, hashlib
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED)
SAMPLE_SIZE = 500_000
IMDB_TOTAL  = 50_000

print('Loading corpus...')
df = pd.read_csv(CSV_PATH, dtype={'label': int})
print(f'Total corpus rows: {len(df):,}')

imdb_df  = df.iloc[:IMDB_TOTAL]
tweet_df = df.iloc[IMDB_TOTAL:]
half            = SAMPLE_SIZE // 2
tweet_pos_slots = half - int((imdb_df['label'] == 1).sum())
tweet_neg_slots = half - int((imdb_df['label'] == 0).sum())
tweet_pos = tweet_df[tweet_df['label'] == 1].sample(n=tweet_pos_slots, random_state=SEED)
tweet_neg = tweet_df[tweet_df['label'] == 0].sample(n=tweet_neg_slots, random_state=SEED)
sample_df = pd.concat([imdb_df, tweet_pos, tweet_neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)
sample_df['text'] = sample_df['text'].astype(str)

texts  = sample_df['text'].tolist()
labels = sample_df['label'].tolist()
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels)
y_test = np.array(y_test)

print(f'Held-out TEST rows: {len(X_test):,}   positives: {y_test.mean():.3f}')
print('Split fingerprint (SHA1, first 1000 test texts):',
      hashlib.sha1('||'.join(X_test[:1000]).encode('utf-8','ignore')).hexdigest()[:16])
print('First test review:', X_test[0][:120])
print('\nThis is the split the served LSTM + DistilBERT held out (they trained on X_train) → leakage-free.')

In [ ]:
# Metrics identical to train_merged_sentiment_models.py (binary, pos_label=1, zero_division=0)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
results = {}
def report(name, y_true, y_pred):
    a = accuracy_score(y_true, y_pred) * 100
    p = precision_score(y_true, y_pred, zero_division=0) * 100
    r = recall_score(y_true, y_pred, zero_division=0) * 100
    f = f1_score(y_true, y_pred, zero_division=0) * 100
    print(f'{name:12s}  Acc {a:5.1f}   Prec {p:5.1f}   Rec {r:5.1f}   F1 {f:5.1f}')
    results[name.lower()] = dict(model=name, accuracy=round(a,1), precision=round(p,1), recall=round(r,1), f1=round(f,1))

In [ ]:
# ---- LSTM ----  Keras tokenizer (CPU) + ONNX inference (GPU), exactly as served
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')   # keep the GPU free for onnxruntime
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np, onnxruntime as ort

MAX_LEN = 100
with open(LSTM_TOKENIZER, encoding='utf-8') as f:
    tok = tokenizer_from_json(f.read())

sess = ort.InferenceSession(LSTM_ONNX, providers=PROVIDERS)
in_name = sess.get_inputs()[0].name
print('LSTM ONNX input:', in_name, sess.get_inputs()[0].shape)

X = pad_sequences(tok.texts_to_sequences(X_test), maxlen=MAX_LEN,
                  padding='post', truncating='post').astype(np.float32)
preds, B = [], 4096
for i in range(0, len(X), B):
    probs = sess.run(None, {in_name: X[i:i+B]})[0].flatten()
    preds.append((probs > 0.5).astype(int))
report('LSTM', y_test, np.concatenate(preds))

In [ ]:
# ---- DistilBERT ----  HF tokenizer + ONNX inference (GPU), max_length=128, argmax (as served)
from transformers import AutoTokenizer
import numpy as np, onnxruntime as ort, glob, os

MAX_LEN = 128
bert_tok = AutoTokenizer.from_pretrained(DISTILBERT_DIR)
onnx_file = os.path.join(DISTILBERT_DIR, 'model.onnx')
if not os.path.exists(onnx_file):
    onnx_file = glob.glob(os.path.join(DISTILBERT_DIR, '*.onnx'))[0]
sess = ort.InferenceSession(onnx_file, providers=PROVIDERS)
input_names = [i.name for i in sess.get_inputs()]
print('DistilBERT ONNX file:', os.path.basename(onnx_file), '| inputs:', input_names)

def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)

preds, B = [], 256
for i in range(0, len(X_test), B):
    enc = bert_tok(X_test[i:i+B], truncation=True, max_length=MAX_LEN,
                   padding=True, return_tensors='np')
    feed = {n: enc[n].astype(np.int64) for n in input_names if n in enc}
    logits = sess.run(None, feed)[0]
    preds.append(softmax(np.asarray(logits)).argmax(axis=1))
report('DistilBERT', y_test, np.concatenate(preds))

In [ ]:
# ---- Summary: paste the two deep-model lines back to Claude ----
print('=== IN-DISTRIBUTION  (held-out 20% of the 500k IMDB+tweets sample, SEED=42, leakage-free) ===\n')
for v in results.values():
    print(f"  {v['model']:12s}  Acc {v['accuracy']:5}  Prec {v['precision']:5}  Rec {v['recall']:5}  F1 {v['f1']:5}")
print('\n--- the 8 classical models on their held-out split, for context (Table 7.2) ---')
print('  LogReg     Acc  77.2  Prec  75.9  Rec  79.8  F1  77.8')
print('  NaiveBayes Acc  75.5  Prec  76.0  Rec  74.4  F1  75.2')
print('  XGBoost    Acc  73.5  Prec  69.9  Rec  82.5  F1  75.7')
print('  RandForest Acc  72.2  Prec  68.2  Rec  83.1  F1  75.0')
print('  MLP        Acc  73.2  Prec  73.3  Rec  72.9  F1  73.1')
print('  DecTree    Acc  69.4  Prec  69.1  Rec  70.3  F1  69.7')
print('  KNN        Acc  66.5  Prec  65.8  Rec  68.6  F1  67.2')
print('  AdaBoost   Acc  57.4  Prec  74.9  Rec  22.4  F1  34.5')
import json; print('\nJSON:', json.dumps(results))